# 05 — Book Q&A Pairs

Reads every chapter from every ARO book and uses the local LLM to generate
questions a developer learning ARO might ask. The answer is drawn directly
from the chapter text — grounded in the authoritative documentation.

Covers all six book collections:
- **TheLanguageGuide** — complete ARO reference (47 chapters + appendices)
- **AROByExample** — hands-on worked examples (14 chapters)
- **TheConstructionStudies** — advanced design patterns
- **TheInteractiveDialog** — conversational explanations
- **ThePluginGuide** — plugin development
- **TheShortStudies** — focused deep-dives
- **Reference** — action/grammar reference

Run **notebook 04** (warm-start fine-tune) after both notebooks 03 and 05 have
appended their pairs, so the adapter benefits from all available knowledge.

**Input:**  `../../Book/` (all `.md` files)
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl` (appended)

In [ ]:
import json, re, sys, random, subprocess
from pathlib import Path
from collections import Counter

def ensure_mlx_lm():
    try:
        from mlx_lm import load, generate as mlx_generate
        from mlx_lm.sample_utils import make_sampler
        return load, mlx_generate, make_sampler
    except ModuleNotFoundError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlx-lm'], check=True)
        from mlx_lm import load, generate as mlx_generate
        from mlx_lm.sample_utils import make_sampler
        return load, mlx_generate, make_sampler

load, mlx_generate, make_sampler = ensure_mlx_lm()

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

ARO_ROOT   = (SCRIPT_DIR / '../../').resolve()
BOOK_ROOT  = ARO_ROOT / 'Book'
DATA_DIR   = SCRIPT_DIR / '../data/02_knowledge'
PAIRS_FILE = DATA_DIR / 'knowledge_pairs.jsonl'

MODEL_ID = 'mlx-community/Qwen2.5-Coder-7B-Instruct-4bit'

# Load knowledge base for system prompt construction
with open(DATA_DIR / 'knowledge.json') as f:
    kb = json.load(f)

print(f'Book root: {BOOK_ROOT}')
print(f'Pairs file: {PAIRS_FILE}')
print(f'Loading {MODEL_ID}...')
model, tokenizer = load(MODEL_ID)
print('Model ready.')

In [ ]:
# ── System prompt ────────────────────────────────────────────────────────────
action_lines = []
for a in kb['actions']:
    if a['verbs']:
        v = '/'.join(a['verbs'][:3])
        p = ', '.join(a['prepositions'][:4])
        action_lines.append(f'- {v}  (role: {a["role"]}, prepositions: {p})')

SYSTEM_PROMPT = f"""You are an expert ARO language teacher creating training data.
ARO (Action Result Object) is a DSL for expressing business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {{
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }}

KEY RULES:
- Articles (a/an/the) are optional
- String concatenation: ++ (NOT +)
- Variable names: hyphenated lowercase e.g. <user-id>
- Forbidden variable name prefixes: is-, with-, empty-
- Exactly ONE Application-Start per application
- openapi.yaml required for HTTP server
- Return statuses: <OK: status>, <Created: status>, <NotFound: status>

AVAILABLE ACTIONS:
{chr(10).join(action_lines[:40])}"""


def chat(messages, max_tokens=800):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return mlx_generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens, verbose=False,
        sampler=make_sampler(temp=0.5),   # slightly higher temp for question diversity
    )


# ── Resume support ───────────────────────────────────────────────────────────
_done_keys = set()
_pair_count = 0
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if line.strip():
                try:
                    s = json.loads(line)
                    _done_keys.add((s['source'], s['instruction'][:80]))
                    _pair_count += 1
                except Exception:
                    pass
    print(f'Resuming — {_pair_count} pairs already in knowledge_pairs.jsonl')


def save_pair(source, instruction, output, score=1.0, metadata=None):
    key = (source, instruction[:80])
    if key in _done_keys:
        return False
    with open(PAIRS_FILE, 'a') as f:
        f.write(json.dumps({
            'instruction': instruction,
            'output':      output,
            'source':      source,
            'score':       score,
            'metadata':    metadata or {},
        }) + '\n')
    _done_keys.add(key)
    return True


print(f'System prompt: {len(SYSTEM_PROMPT)} chars')
print('Helpers ready.')

## Discover Book Chapters

Collect every `.md` file from every book collection.

In [ ]:
# Each entry: {book, chapter_name, path, content}
all_chapters = []

for book_dir in sorted(BOOK_ROOT.iterdir()):
    if not book_dir.is_dir():
        continue
    book_name = book_dir.name
    md_files  = sorted(book_dir.glob('*.md'))
    for md_path in md_files:
        content = md_path.read_text().strip()
        if len(content) < 200:   # skip near-empty stubs
            continue
        all_chapters.append({
            'book':    book_name,
            'name':    md_path.stem,
            'path':    md_path,
            'content': content,
        })
    print(f'  {book_name}: {len(md_files)} files')

print(f'\nTotal chapters to process: {len(all_chapters)}')

## Q&A Extraction Loop

For each chapter:
1. Split into sections by heading (`##`, `###`)
2. For each substantive section, ask the LLM to generate 2–3 questions a developer might ask
3. The answer is drawn directly from the section text (+ optional LLM polish)
4. Save each (question, answer) pair to `knowledge_pairs.jsonl`

This teaches the model *how to explain* ARO concepts in prose — complementing
the code-generation pairs from notebooks 02 and 03.

In [ ]:
def split_into_sections(content):
    """
    Split a markdown document into (heading, body) pairs.
    Top-level chapter prose (before any heading) is included as heading=chapter_title.
    """
    # Extract chapter title from first # heading
    title_match = re.search(r'^#\s+(.+)', content, re.MULTILINE)
    chapter_title = title_match.group(1).strip() if title_match else 'Introduction'

    # Split on ## or ### headings
    parts = re.split(r'(?m)^(#{2,3}\s+.+)$', content)

    sections = []
    # First part: content before first ## heading
    if parts[0].strip() and len(parts[0].strip()) > 100:
        sections.append((chapter_title, parts[0].strip()))

    # Subsequent heading+body pairs
    i = 1
    while i < len(parts) - 1:
        heading = re.sub(r'^#{2,3}\s+', '', parts[i]).strip()
        body    = parts[i + 1].strip() if i + 1 < len(parts) else ''
        if body and len(body) > 120:   # skip very short sections
            sections.append((heading, body))
        i += 2

    return sections


def generate_qa_pairs(book, chapter_name, heading, section_body):
    """
    Ask the LLM to generate Q&A pairs from a book section.
    Returns list of (question, answer) tuples.
    """
    # Trim section to fit context window
    body_trimmed = section_body[:3000]

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'You are creating Q&A training pairs from the ARO documentation.\n\n'
            f'Book: {book}\n'
            f'Chapter: {chapter_name}\n'
            f'Section: {heading}\n\n'
            f'--- Section Content ---\n{body_trimmed}\n--- End ---\n\n'
            f'Generate 3 questions a developer learning ARO might ask after reading '
            f'this section. For each question, write a clear answer based on the '
            f'section content above. Include any relevant ARO code examples.\n\n'
            f'Format EXACTLY like this:\n\n'
            f'### Q1\n'
            f'**Question:** <question>\n'
            f'**Answer:** <answer drawn from the section>\n\n'
            f'### Q2\n'
            f'**Question:** ...\n'
            f'**Answer:** ...\n\n'
            f'### Q3\n'
            f'**Question:** ...\n'
            f'**Answer:** ...'
        )},
    ]

    output = chat(messages, max_tokens=900)

    # Parse Q&A blocks
    pairs = []
    blocks = re.split(r'###\s*Q\d+', output)
    for block in blocks[1:]:
        q_match = re.search(
            r'\*\*Question:\*\*\s*(.+?)(?=\n\*\*Answer|\Z)', block, re.DOTALL
        )
        a_match = re.search(
            r'\*\*Answer:\*\*\s*(.+?)(?=\n###|\Z)', block, re.DOTALL
        )
        if q_match and a_match:
            question = q_match.group(1).strip().replace('\n', ' ')
            answer   = a_match.group(1).strip()
            if len(question) > 10 and len(answer) > 20:
                pairs.append((question, answer))
    return pairs


# ── Main loop ────────────────────────────────────────────────────────────────
print(f'\n--- Q&A Extraction: {len(all_chapters)} chapters ---')
total_new = 0

for ch in all_chapters:
    book         = ch['book']
    chapter_name = ch['name']
    content      = ch['content']
    source_prefix = f'book_qa:{book}:{chapter_name}'

    sections = split_into_sections(content)
    chapter_new = 0

    for heading, body in sections:
        source_key = f'{source_prefix}:{heading[:50]}'

        # Skip if all Q&As from this section are already done
        if all((_done_keys and (source_key, f'q{i}') in _done_keys)
               for i in range(3)):
            continue

        pairs = generate_qa_pairs(book, chapter_name, heading, body)
        for question, answer in pairs:
            if save_pair(
                source_key,
                question,
                answer,
                score=1.0,
                metadata={'book': book, 'chapter': chapter_name, 'section': heading},
            ):
                chapter_new += 1
                total_new   += 1

    if chapter_new > 0:
        print(f'  {book}/{chapter_name}: +{chapter_new} pairs', flush=True)

print(f'\nDone: {total_new} new Q&A pairs added')

## Summary

In [ ]:
all_pairs = []
with open(PAIRS_FILE) as f:
    for line in f:
        if line.strip():
            try:
                all_pairs.append(json.loads(line))
            except Exception:
                pass

sources = Counter(p['source'].split(':')[0] for p in all_pairs)

print(f'Total pairs in knowledge_pairs.jsonl: {len(all_pairs)}')
print('\nBy source type:')
for src, n in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {src:30s}: {n}')

qa_pairs = [p for p in all_pairs if p['source'].startswith('book_qa:')]
books    = Counter(p['source'].split(':')[1] for p in qa_pairs)
print(f'\nQ&A pairs by book ({len(qa_pairs)} total):')
for book, n in sorted(books.items(), key=lambda x: -x[1]):
    print(f'  {book:35s}: {n}')